In [ ]:
%load_ext cash
%cash_on
%cash_badge print
%cash_debug on

# Project 4: US Census ACS Demographic Analysis

**Goal**: Analyze American Community Survey (ACS) Public Use Microdata Samples (PUMS) to study income inequality, education-employment correlations, housing cost burden, and geographic patterns across US states.

**Data**: US Census Bureau ACS 1-Year PUMS 2022 (person-level + housing-level records)  
**Source**: https://www2.census.gov/programs-surveys/acs/data/pums/2022/1-Year/  
**Size**: ~350-500 MB (CSV.zip person file + housing file)

**Cash Stress Points**:
- Two large CSV files with interdependent analysis
- Survey weighting calculations (PWGTP / WGTP replicate weights)
- Complex cross-tabulations and conditional analysis
- Multiple derived columns and chained transformations
- Geographic aggregation at different levels (state, PUMA)

In [ ]:
import os
import time
import urllib.request
import zipfile

In [ ]:
# Download ACS PUMS 2022 1-Year person-level data (all states)
# The person file has ~3.2M records with 500+ variables
data_dir = os.path.join(os.getcwd(), 'examples', 'large_scale_projects', 'data', 'census_acs')
os.makedirs(data_dir, exist_ok=True)

# ACS PUMS files from Census Bureau
files_to_download = {
    'csv_pus.zip': 'https://www2.census.gov/programs-surveys/acs/data/pums/2022/1-Year/csv_pus.zip',
    'csv_hus.zip': 'https://www2.census.gov/programs-surveys/acs/data/pums/2022/1-Year/csv_hus.zip',
}

opener = urllib.request.build_opener()
opener.addheaders = [('User-Agent', 'Mozilla/5.0 (Cash-Benchmark/1.0; research project)')]
urllib.request.install_opener(opener)

t0 = time.time()
for fname, url in files_to_download.items():
    fpath = os.path.join(data_dir, fname)
    if os.path.exists(fpath):
        sz = os.path.getsize(fpath) / 1e6
        print(f"Already downloaded: {fname} ({sz:.1f} MB)")
    else:
        print(f"Downloading {fname}...")
        try:
            urllib.request.urlretrieve(url, fpath)
            sz = os.path.getsize(fpath) / 1e6
            elapsed = time.time() - t0
            print(f"  Downloaded: {fname} ({sz:.1f} MB, {elapsed:.0f}s)")
        except Exception as e:
            print(f"  FAILED: {e}")

# Extract CSV files from zips
for fname in files_to_download.keys():
    zpath = os.path.join(data_dir, fname)
    if os.path.exists(zpath):
        with zipfile.ZipFile(zpath, 'r') as zf:
            csv_names = [n for n in zf.namelist() if n.endswith('.csv')]
            for csv_name in csv_names:
                csv_path = os.path.join(data_dir, csv_name)
                if not os.path.exists(csv_path):
                    print(f"  Extracting {csv_name}...")
                    zf.extract(csv_name, data_dir)
                else:
                    print(f"  Already extracted: {csv_name}")

elapsed = time.time() - t0
print(f"\nSetup complete in {elapsed:.0f}s")

# List all CSV files
csv_files = [f for f in os.listdir(data_dir) if f.endswith('.csv')]
total_csv_size = sum(os.path.getsize(os.path.join(data_dir, f)) for f in csv_files)
print(f"CSV files: {csv_files}")
print(f"Total CSV size: {total_csv_size / 1e9:.2f} GB ({total_csv_size / 1e6:.0f} MB)")

In [ ]:
# Load person-level PUMS data with selected columns
# Full file has 500+ columns - we select the ones relevant for our analysis
import pandas as _pd4
import numpy as _np4
import os as _os4
import time as _time4

_data_dir = _os4.path.join(_os4.getcwd(), 'examples', 'large_scale_projects', 'data', 'census_acs')

# Key PUMS person variables for our analysis:
# SERIALNO - Housing unit serial number (links to housing file)
# SPORDER - Person number within housing unit
# PWGTP - Person weight
# ST - State FIPS code
# AGEP - Age
# SEX - Sex (1=Male, 2=Female)
# RAC1P - Race (recoded)
# HISP - Hispanic origin
# SCHL - Educational attainment
# ESR - Employment status
# COW - Class of worker
# WAGP - Wages/salary income
# PINCP - Total personal income
# WKHP - Hours worked per week
# JWMNP - Travel time to work (minutes)
# NAICSP - NAICS industry code
# SOCP - SOC occupation code
# PUMA - Public Use Microdata Area

_person_cols = ['SERIALNO', 'SPORDER', 'PWGTP', 'ST', 'AGEP', 'SEX', 'RAC1P', 
                'HISP', 'SCHL', 'ESR', 'COW', 'WAGP', 'PINCP', 'WKHP',
                'JWMNP', 'NAICSP', 'SOCP', 'PUMA']

_t0 = _time4.time()

# Find person CSV files (there may be multiple parts: psam_pusa.csv, psam_pusb.csv)
_person_csvs = sorted([f for f in _os4.listdir(_data_dir) 
                        if f.startswith('psam_pus') and f.endswith('.csv')])
print(f"Person CSV files: {_person_csvs}")

_dfs = []
for _csv in _person_csvs:
    _path = _os4.path.join(_data_dir, _csv)
    _sz = _os4.path.getsize(_path) / 1e6
    print(f"Reading {_csv} ({_sz:.0f} MB)...")
    _df_part = _pd4.read_csv(_path, usecols=_person_cols, dtype={'NAICSP': str, 'SOCP': str, 'SERIALNO': str})
    _dfs.append(_df_part)
    print(f"  {len(_df_part):,} records")

_df = _pd4.concat(_dfs, ignore_index=True)
_elapsed = _time4.time() - _t0
print(f"\nLoaded {len(_df):,} person records in {_elapsed:.1f}s")
print(f"Memory: {_df.memory_usage(deep=True).sum() / 1e6:.0f} MB")
print(f"Columns: {list(_df.columns)}")
print(f"\nWeighted population: {_df['PWGTP'].sum():,.0f}")
print(f"States represented: {_df['ST'].nunique()}")
print(f"Age range: {_df['AGEP'].min()} - {_df['AGEP'].max()}")

In [ ]:
# ── Cell 5: State-level income analysis (weighted) ──
# Map FIPS codes to state abbreviations
_fips_to_state = {
    1: 'AL', 2: 'AK', 4: 'AZ', 5: 'AR', 6: 'CA', 8: 'CO', 9: 'CT', 10: 'DE',
    11: 'DC', 12: 'FL', 13: 'GA', 15: 'HI', 16: 'ID', 17: 'IL', 18: 'IN', 19: 'IA',
    20: 'KS', 21: 'KY', 22: 'LA', 23: 'ME', 24: 'MD', 25: 'MA', 26: 'MI', 27: 'MN',
    28: 'MS', 29: 'MO', 30: 'MT', 31: 'NE', 32: 'NV', 33: 'NH', 34: 'NJ', 35: 'NM',
    36: 'NY', 37: 'NC', 38: 'ND', 39: 'OH', 40: 'OK', 41: 'OR', 42: 'PA', 44: 'RI',
    45: 'SC', 46: 'SD', 47: 'TN', 48: 'TX', 49: 'UT', 50: 'VT', 51: 'VA', 53: 'WA',
    54: 'WV', 55: 'WI', 56: 'WY', 72: 'PR'
}

_df['state'] = _df['ST'].map(_fips_to_state)

# Working-age population (25-64) with income > 0
_workers = _df[(_df['AGEP'] >= 25) & (_df['AGEP'] <= 64) & (_df['PINCP'] > 0)].copy()
print(f"Working-age population with income: {len(_workers):,} records")
print(f"Weighted: {_workers['PWGTP'].sum():,.0f}")

# Weighted median income by state
def _weighted_median(data, weights):
    """Calculate weighted median."""
    sorted_idx = _np4.argsort(data)
    sorted_data = data[sorted_idx]
    sorted_weights = weights[sorted_idx]
    cum_weight = _np4.cumsum(sorted_weights)
    cutoff = sorted_weights.sum() / 2.0
    return sorted_data[cum_weight >= cutoff][0]

_state_income = {}
for _st in _workers['state'].dropna().unique():
    _st_data = _workers[_workers['state'] == _st]
    _med_income = _weighted_median(_st_data['PINCP'].values, _st_data['PWGTP'].values)
    _mean_income = _np4.average(_st_data['PINCP'].values, weights=_st_data['PWGTP'].values)
    _state_income[_st] = {'median': _med_income, 'mean': _mean_income, 
                           'n_records': len(_st_data), 'pop': _st_data['PWGTP'].sum()}

_state_df = _pd4.DataFrame(_state_income).T
_state_df = _state_df.sort_values('median', ascending=False)
print(f"\nTop 10 states by median personal income (weighted):")
_top10 = _state_df.head(10)
for _rank in range(len(_top10)):
    _st = _top10.index[_rank]
    _row_vals = _top10.iloc[_rank]
    print(f"  {_rank+1}. {_st}: ${_row_vals['median']:,.0f} (mean: ${_row_vals['mean']:,.0f}, pop: {_row_vals['pop']:,.0f})")

print(f"\nBottom 5 states:")
_bot5 = _state_df.tail(5)
for _rank in range(len(_bot5)):
    _st = _bot5.index[_rank]
    _row_vals = _bot5.iloc[_rank]
    print(f"  {_st}: ${_row_vals['median']:,.0f} (mean: ${_row_vals['mean']:,.0f})")

In [ ]:
# ── Cell 6: Education-Income correlation ──
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Education level mapping (SCHL codes)
_edu_map = {
    1: 'No schooling', 2: 'Nursery-4th', 3: 'Nursery-4th', 4: 'Nursery-4th',
    5: 'Nursery-4th', 6: '5th-8th', 7: '5th-8th', 8: '5th-8th', 9: '5th-8th',
    10: '9th', 11: '10th', 12: '11th', 13: '12th no diploma',
    14: 'Regular HS diploma', 15: 'GED/alternative', 16: 'Some college <1yr',
    17: 'Some college 1+yr', 18: "Associate's", 19: "Associate's",
    20: "Bachelor's", 21: "Master's", 22: "Professional", 23: "Doctorate", 24: "Doctorate"
}

# Simplified education categories
_edu_simple = {
    1: 'Less than HS', 2: 'Less than HS', 3: 'Less than HS', 4: 'Less than HS',
    5: 'Less than HS', 6: 'Less than HS', 7: 'Less than HS', 8: 'Less than HS', 
    9: 'Less than HS', 10: 'Less than HS', 11: 'Less than HS', 12: 'Less than HS',
    13: 'Less than HS',
    14: 'High School', 15: 'High School',
    16: 'Some College', 17: 'Some College', 18: "Associate's", 19: "Associate's",
    20: "Bachelor's", 21: "Master's", 22: "Professional+", 23: "Doctorate", 24: "Doctorate"
}

_workers['edu_level'] = _workers['SCHL'].map(_edu_simple)

_edu_order = ['Less than HS', 'High School', 'Some College', "Associate's", 
              "Bachelor's", "Master's", "Professional+", "Doctorate"]

_edu_income = {}
for _edu in _edu_order:
    _sub = _workers[_workers['edu_level'] == _edu]
    if len(_sub) > 0:
        _med = _weighted_median(_sub['PINCP'].values, _sub['PWGTP'].values)
        _mean = _np4.average(_sub['PINCP'].values, weights=_sub['PWGTP'].values)
        _edu_income[_edu] = {'median': _med, 'mean': _mean, 'pop': _sub['PWGTP'].sum()}

_edu_df = _pd4.DataFrame(_edu_income).T

_fig1, _ax1 = plt.subplots(figsize=(10, 6))
_x = range(len(_edu_df))
_ax1.bar(_x, _edu_df['median'], color='steelblue', alpha=0.8, label='Median')
_ax1.bar(_x, _edu_df['mean'], color='coral', alpha=0.4, label='Mean')
_ax1.set_xticks(_x)
_ax1.set_xticklabels(_edu_df.index, rotation=45, ha='right')
_ax1.set_ylabel('Personal Income ($)')
_ax1.set_title('Income by Education Level (ACS 2022, Ages 25-64)')
_ax1.legend()
_ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))
plt.tight_layout()
plt.savefig('examples/large_scale_projects/data/census_acs/education_income.png', dpi=150)
plt.show()

print("Education-Income Summary (Weighted):")
for _edu in _edu_order:
    if _edu in _edu_income:
        _info = _edu_income[_edu]
        print(f"  {_edu:20s}: median ${_info['median']:>8,.0f}, mean ${_info['mean']:>8,.0f}, pop {_info['pop']:>12,.0f}")

In [ ]:
# ── Cell 7: Age-Income profile by sex ──
_workers['age_group'] = _pd4.cut(_workers['AGEP'], bins=[24, 29, 34, 39, 44, 49, 54, 59, 65],
                                  labels=['25-29', '30-34', '35-39', '40-44', '45-49', '50-54', '55-59', '60-64'])
_workers['sex_label'] = _workers['SEX'].map({1: 'Male', 2: 'Female'})

_age_sex_income = {}
for _sex in ['Male', 'Female']:
    _age_sex_income[_sex] = {}
    for _ag in ['25-29', '30-34', '35-39', '40-44', '45-49', '50-54', '55-59', '60-64']:
        _sub = _workers[(_workers['sex_label'] == _sex) & (_workers['age_group'] == _ag)]
        if len(_sub) > 0:
            _med = _weighted_median(_sub['PINCP'].values, _sub['PWGTP'].values)
            _age_sex_income[_sex][_ag] = _med

_fig2, _ax2 = plt.subplots(figsize=(10, 6))
_ages = ['25-29', '30-34', '35-39', '40-44', '45-49', '50-54', '55-59', '60-64']
_male_inc = [_age_sex_income['Male'].get(_a, 0) for _a in _ages]
_female_inc = [_age_sex_income['Female'].get(_a, 0) for _a in _ages]

_ax2.plot(_ages, _male_inc, 'o-', color='steelblue', linewidth=2, label='Male', markersize=8)
_ax2.plot(_ages, _female_inc, 's-', color='coral', linewidth=2, label='Female', markersize=8)
_ax2.set_xlabel('Age Group')
_ax2.set_ylabel('Median Personal Income ($)')
_ax2.set_title('Age-Income Profile by Sex (ACS 2022, Workers with Income)')
_ax2.legend()
_ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))
_ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('examples/large_scale_projects/data/census_acs/age_income_by_sex.png', dpi=150)
plt.show()

# Gender pay gap calculation
_total_male_med = _weighted_median(_workers[_workers['sex_label'] == 'Male']['PINCP'].values,
                                    _workers[_workers['sex_label'] == 'Male']['PWGTP'].values)
_total_female_med = _weighted_median(_workers[_workers['sex_label'] == 'Female']['PINCP'].values,
                                      _workers[_workers['sex_label'] == 'Female']['PWGTP'].values)
_gap = (_total_male_med - _total_female_med) / _total_male_med * 100
print(f"Overall median income - Male: ${_total_male_med:,.0f}, Female: ${_total_female_med:,.0f}")
print(f"Gender pay gap: {_gap:.1f}% (women earn {100-_gap:.1f} cents per dollar)")

In [ ]:
# ── Cell 8: Housing cost burden analysis ──
# Load housing file to analyze rent/mortgage burden

_housing_cols = ['SERIALNO', 'WGTP', 'ST', 'TEN', 'GRNTP', 'SMOCP', 'HINCP', 
                 'NP', 'BLD', 'BDSP', 'RMSP', 'VALP']

_housing_csvs = sorted([f for f in _os4.listdir(_data_dir) 
                         if f.startswith('psam_hus') and f.endswith('.csv')])
print(f"Housing CSV files: {_housing_csvs}")

_h_dfs = []
_t0h = _time4.time()
for _csv in _housing_csvs:
    _path = _os4.path.join(_data_dir, _csv)
    _sz = _os4.path.getsize(_path) / 1e6
    print(f"Reading {_csv} ({_sz:.0f} MB)...")
    _h_part = _pd4.read_csv(_path, usecols=_housing_cols, dtype={'SERIALNO': str})
    _h_dfs.append(_h_part)
    print(f"  {len(_h_part):,} records")

_hdf = _pd4.concat(_h_dfs, ignore_index=True)
_elapsed_h = _time4.time() - _t0h
print(f"\nLoaded {len(_hdf):,} housing records in {_elapsed_h:.1f}s")
print(f"Memory: {_hdf.memory_usage(deep=True).sum() / 1e6:.0f} MB")

_hdf['state'] = _hdf['ST'].map(_fips_to_state)

# Housing cost burden: gross rent or mortgage > 30% of household income
# TEN: 1=Owned w/mortgage, 2=Owned free, 3=Rented
# GRNTP: Gross rent (monthly), SMOCP: Selected monthly owner costs (mortgage)
# HINCP: Household income

# For renters
_renters = _hdf[(_hdf['TEN'] == 3) & (_hdf['GRNTP'] > 0) & (_hdf['HINCP'] > 0)].copy()
_renters['cost_ratio'] = (_renters['GRNTP'] * 12) / _renters['HINCP']
_renters['burdened'] = _renters['cost_ratio'] > 0.30
_renters['severely_burdened'] = _renters['cost_ratio'] > 0.50

# For owners with mortgage
_owners_m = _hdf[(_hdf['TEN'] == 1) & (_hdf['SMOCP'] > 0) & (_hdf['HINCP'] > 0)].copy()
_owners_m['cost_ratio'] = (_owners_m['SMOCP'] * 12) / _owners_m['HINCP']
_owners_m['burdened'] = _owners_m['cost_ratio'] > 0.30
_owners_m['severely_burdened'] = _owners_m['cost_ratio'] > 0.50

# Weighted burden rates
_renter_burden = _np4.average(_renters['burdened'].values, weights=_renters['WGTP'].values) * 100
_owner_burden = _np4.average(_owners_m['burdened'].values, weights=_owners_m['WGTP'].values) * 100
_renter_severe = _np4.average(_renters['severely_burdened'].values, weights=_renters['WGTP'].values) * 100
_owner_severe = _np4.average(_owners_m['severely_burdened'].values, weights=_owners_m['WGTP'].values) * 100

print(f"\nHousing Cost Burden (>30% of income):")
print(f"  Renters: {_renter_burden:.1f}% burdened, {_renter_severe:.1f}% severely burdened (>50%)")
print(f"  Owners w/mortgage: {_owner_burden:.1f}% burdened, {_owner_severe:.1f}% severely burdened")

# State-level renter burden
_state_burden = {}
for _st in _renters['state'].dropna().unique():
    _st_r = _renters[_renters['state'] == _st]
    if len(_st_r) > 50:
        _b = _np4.average(_st_r['burdened'].values, weights=_st_r['WGTP'].values) * 100
        _state_burden[_st] = _b

_burden_series = _pd4.Series(_state_burden).sort_values(ascending=False)
print(f"\nTop 10 states by renter cost burden:")
for _rank in range(min(10, len(_burden_series))):
    _st = _burden_series.index[_rank]
    _b = _burden_series.iloc[_rank]
    print(f"  {_st}: {_b:.1f}%")

In [ ]:
# ── Cell 9: Race/ethnicity income disparities ──
_race_map = {1: 'White', 2: 'Black', 3: 'Native American', 4: 'Native American',
             5: 'Native American', 6: 'Asian', 7: 'Pacific Islander', 
             8: 'Other', 9: 'Two or more'}
_workers['race'] = _workers['RAC1P'].map(_race_map)
_workers['hispanic'] = _workers['HISP'].apply(lambda x: 'Hispanic' if x > 1 else 'Non-Hispanic')

# Create race/ethnicity categories
_workers['race_eth'] = _workers.apply(
    lambda r: 'Hispanic (any race)' if r['hispanic'] == 'Hispanic' 
    else r['race'] if r['race'] in ['White', 'Black', 'Asian']
    else 'Other', axis=1)

_race_categories = ['White', 'Black', 'Asian', 'Hispanic (any race)', 'Other']
_race_income = {}
for _cat in _race_categories:
    _sub = _workers[_workers['race_eth'] == _cat]
    if len(_sub) > 100:
        _med = _weighted_median(_sub['PINCP'].values, _sub['PWGTP'].values)
        _mean = _np4.average(_sub['PINCP'].values, weights=_sub['PWGTP'].values)
        _race_income[_cat] = {'median': _med, 'mean': _mean, 'pop': _sub['PWGTP'].sum()}

_fig3, _ax3 = plt.subplots(figsize=(10, 6))
_race_df = _pd4.DataFrame(_race_income).T
_x3 = range(len(_race_df))
_bars = _ax3.bar(_x3, _race_df['median'], color=['steelblue', 'coral', 'seagreen', 'goldenrod', 'mediumpurple'])
_ax3.set_xticks(_x3)
_ax3.set_xticklabels(_race_df.index, rotation=30, ha='right')
_ax3.set_ylabel('Median Personal Income ($)')
_ax3.set_title('Income by Race/Ethnicity (ACS 2022, Ages 25-64)')
_ax3.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))
for _i3, _v3 in enumerate(_race_df['median']):
    _ax3.text(_i3, _v3 + 500, f'${_v3:,.0f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('examples/large_scale_projects/data/census_acs/race_income.png', dpi=150)
plt.show()

print("Race/Ethnicity Income Summary:")
for _cat in _race_categories:
    if _cat in _race_income:
        _info = _race_income[_cat]
        print(f"  {_cat:25s}: median ${_info['median']:>8,.0f}, mean ${_info['mean']:>8,.0f}, pop {_info['pop']:>12,.0f}")

In [ ]:
# ── Cell 10: Commute patterns ──
_commuters = _workers[(_workers['JWMNP'] > 0) & (_workers['JWMNP'] < 200)].copy()
print(f"Commuters with valid travel time: {len(_commuters):,}")

# Weighted commute time distribution
_commute_bins = [0, 10, 20, 30, 45, 60, 90, 120, 200]
_commute_labels = ['<10', '10-20', '20-30', '30-45', '45-60', '60-90', '90-120', '120+']
_commuters['commute_bin'] = _pd4.cut(_commuters['JWMNP'], bins=_commute_bins, labels=_commute_labels)

_commute_dist = {}
for _bin in _commute_labels:
    _sub = _commuters[_commuters['commute_bin'] == _bin]
    if len(_sub) > 0:
        _commute_dist[_bin] = _sub['PWGTP'].sum()

_fig4, _ax4 = plt.subplots(figsize=(10, 5))
_cd_series = _pd4.Series(_commute_dist)
_cd_pct = _cd_series / _cd_series.sum() * 100
_cd_pct.plot(kind='bar', ax=_ax4, color='teal', alpha=0.8)
_ax4.set_xlabel('Commute Time (minutes)')
_ax4.set_ylabel('% of Commuters')
_ax4.set_title('Commute Time Distribution (ACS 2022, Workers)')
_ax4.tick_params(axis='x', rotation=0)
for _i4, _v4 in enumerate(_cd_pct):
    _ax4.text(_i4, _v4 + 0.5, f'{_v4:.1f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('examples/large_scale_projects/data/census_acs/commute_distribution.png', dpi=150)
plt.show()

# Weighted mean commute by state
_state_commute = {}
for _st in _commuters['state'].dropna().unique():
    _sub = _commuters[_commuters['state'] == _st]
    if len(_sub) > 50:
        _mean_c = _np4.average(_sub['JWMNP'].values, weights=_sub['PWGTP'].values)
        _state_commute[_st] = _mean_c

_commute_series = _pd4.Series(_state_commute).sort_values(ascending=False)
print(f"\nTop 10 states by mean commute time:")
for _st, _c in _commute_series.head(10).items():
    print(f"  {_st}: {_c:.1f} min")
print(f"\nShortest commutes:")
for _st, _c in _commute_series.tail(5).items():
    print(f"  {_st}: {_c:.1f} min")

In [ ]:
# ── Cell 11: Summary statistics ──
print("=" * 60)
print("PROJECT 4: US Census ACS Demographic Analysis - Summary")
print("=" * 60)
print(f"\nDataset: ACS 2022 1-Year PUMS")
print(f"Person records: {len(_df):,}")
print(f"Housing records: {len(_hdf):,}")
print(f"Weighted population (25-64 w/income): {_workers['PWGTP'].sum():,.0f}")
print(f"Person file memory: {_df.memory_usage(deep=True).sum() / 1e6:.0f} MB")
print(f"Housing file memory: {_hdf.memory_usage(deep=True).sum() / 1e6:.0f} MB")
print(f"\nKey Findings:")
print(f"  - Gender pay gap: {_gap:.1f}%")
print(f"  - Renter cost burden rate: {_renter_burden:.1f}%")
print(f"  - Top income state: {_state_df.index[0]} (median ${_state_df.iloc[0]['median']:,.0f})")
print(f"  - Bottom income state: {_state_df.index[-1]} (median ${_state_df.iloc[-1]['median']:,.0f})")
print(f"  - Education premium: Bachelor's vs HS")
_bach = _edu_income.get("Bachelor's", {}).get('median', 0)
_hs = _edu_income.get("High School", {}).get('median', 0)
if _hs > 0:
    print(f"    Bachelor's median: ${_bach:,.0f}, HS median: ${_hs:,.0f} ({_bach/_hs:.1f}x)")
print(f"\nCash Performance:")
print(f"  - Person file load time: {_elapsed:.1f}s")
print(f"  - Housing file load time: {_elapsed_h:.1f}s")
print(f"  - Weighted calculations on {len(_workers):,} records")
print("=" * 60)